<a href="https://colab.research.google.com/github/tburleyinfo/vLLM-Hook/blob/codex/MLR-45-cnl-spotlight-blocking-colab/notebooks/demo_cnl_spotlight_blocking_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Controlled Natural Language With Spotlight And Logit Blocking In Colab

This notebook demonstrates a lightweight Controlled Natural Language (CNL) example. It combines Spotlight promotion for preferred wording with vLLM `bad_words` logit blocking for words the generation should avoid. The checks use spaCy so the results stay easy to inspect.

The goal is simple English style control, not full formal CNL parsing.


### Installation

Run this setup cell once in a fresh Colab GPU runtime before continuing. It clones the repo and installs the CUDA-compatible notebook dependencies.


In [1]:
# ==============================================================================
# VLLM HOOK SETUP AND DEPENDENCY MANAGER
# ==============================================================================
#
# PURPOSE:
# This cell prepares the environment to run vLLM and its custom plugins.
# It handles complex dependency conflicts common in Colab (e.g., pre-installed
# torch versions) and ensures the plugin code is loaded correctly.
#
# KEY ACTIONS:
# 1. Clones the vLLM-Hook repository if not present.
# 2. Installs a compatible version of vLLM and PyTorch for your GPU.
# 3. Cleans up old binary artifacts to prevent version conflicts.
# 4. Loads the plugin source code.
# 5. TRIGGERS A COLAB RESTART: The cell will restart the runtime to ensure
#    the new libraries are fully loaded into the kernel memory.
#
# EXPECTED BEHAVIOR:
# - The cell will run and install packages.
# - It may print "Restarting Colab runtime...".
# - The cell will stop abruptly, and the runtime will restart.
# - The restart is automatic; the code will resume from the top.
# ==============================================================================

from pathlib import Path
import importlib
import importlib.util
import os
import re
import shutil
import site
import subprocess
import sys
import time

# Configuration
REPO_URL = os.environ.get("VLLM_HOOK_REPO_URL", "https://github.com/tburleyinfo/vLLM-Hook.git")
REPO_BRANCH = os.environ.get("VLLM_HOOK_REPO_BRANCH", "codex/MLR-45-cnl-spotlight-blocking-colab")
REPO_NAME = "vLLM-Hook"

# Environment Variables (Optional overrides)
COLAB_INSTALL_VLLM = os.environ.get("COLAB_INSTALL_VLLM", "")
VLLM_SPEC = os.environ.get("VLLM_SPEC", "vllm>=0.11,<0.19")
VLLM_TORCH_BACKEND = os.environ.get("VLLM_TORCH_BACKEND", "cu128")

IN_COLAB = "google.colab" in sys.modules
NOTEBOOK_DIR = Path.cwd()

# --------------------------------------------------------------------------
# Helper Functions
# --------------------------------------------------------------------------

def run(cmd, cwd=None, env=None):
    """
    Executes a shell command, printing output in real-time.
    Raises an error if the command fails.
    """
    cmd = [str(part) for part in cmd]
    print(f"> Running: {' '.join(cmd)}", flush=True)

    process = subprocess.Popen(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    tail = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
        tail = tail[-120:]

    returncode = process.wait()
    if returncode:
        tail_text = "\n".join(tail)
        raise RuntimeError(
            f"Command failed with exit code {returncode}: {' '.join(cmd)}\n\n"
            f"Last output lines:\n{tail_text}"
        )

def run_capture(cmd):
    """Executes a command and returns stdout/stderr without printing."""
    return subprocess.run([str(part) for part in cmd], text=True, capture_output=True, check=False)

def norm(name):
    """Normalize package names for comparison."""
    return name.lower().replace("_", "-")

def package_from_req_line(line: str) -> str:
    """Extract package name from a requirement string (e.g., 'torch>=1.0' -> 'torch')."""
    stripped = line.strip()
    # Split on version specifiers
    package = re.split(r"==|>=|<=|~=|!=|<|>|\[", stripped, maxsplit=1)[0]
    return norm(package.strip())

def _repo_remote_matches(repo_root: Path, expected_remote: str) -> bool:
    """Checks if the git repo's origin matches the expected URL."""
    try:
        url = subprocess.run(
            ["git", "-C", str(repo_root), "remote", "get-url", "origin"],
            check=True,
            capture_output=True,
            text=True,
        ).stdout.strip().removesuffix(".git")
    except Exception:
        return False
    return url == expected_remote

def _find_existing_repo_root(start_dir: Path, expected_remote: str):
    """Searches up the directory tree for a matching git repo."""
    for candidate in [start_dir, *start_dir.parents]:
        if (candidate / ".git").exists() and _repo_remote_matches(candidate, expected_remote):
            return candidate
    return None

def assert_cuda_runtime():
    """Ensures a GPU is available before proceeding."""
    try:
        import torch
    except Exception:
        torch = None

    has_cuda = bool(torch is not None and torch.cuda.is_available())
    has_cudart = importlib.util.find_spec("nvidia.cuda_runtime") is not None

    if not has_cuda and not has_cudart:
        raise RuntimeError(
            "No CUDA GPU detected. "
            "In Colab, go to Runtime > Change runtime type and select T4 GPU (or better), "
            "then re-run the entire notebook from the beginning."
        )

# --------------------------------------------------------------------------
# 1. Repository Setup
# --------------------------------------------------------------------------

expected_remote = REPO_URL.removesuffix(".git")
existing_repo_root = _find_existing_repo_root(NOTEBOOK_DIR, expected_remote)

if IN_COLAB:
    if existing_repo_root is not None:
        REPO_ROOT = existing_repo_root
        print(f"Refreshing existing repo at: {REPO_ROOT}")
        run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
        run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])
    else:
        REPO_ROOT = Path("/content") / REPO_NAME
        if not REPO_ROOT.exists():
            print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        elif not _repo_remote_matches(REPO_ROOT, expected_remote):
            print(f"Remote mismatch detected. Replacing clone...")
            shutil.rmtree(REPO_ROOT)
            run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
        else:
            print(f"Refreshing existing clone...")
            run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH])
            run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only", "origin", REPO_BRANCH])

    NOTEBOOK_DIR = REPO_ROOT / "notebooks"
    os.chdir(NOTEBOOK_DIR)
    print(f"Working directory set to: {NOTEBOOK_DIR}")
else:
    REPO_ROOT = NOTEBOOK_DIR.parent

PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
FILTERED_REQ_FILE = Path("/tmp/vllm_hook_colab_requirements.txt")
COLAB_RESTART_MARKER = Path("/tmp/vllm_hook_colab_binary_deps_restarted")

print(f"\n--- Environment Summary ---")
print(f"Running in Colab: {IN_COLAB}")
print(f"Repo Root: {REPO_ROOT}")
print(f"Plugin Dir: {PKG_DIR}")

if IN_COLAB:
    assert_cuda_runtime()

# --------------------------------------------------------------------------
# 2. Plugin Directory Validation
# --------------------------------------------------------------------------

if not PKG_DIR.exists():
    raise FileNotFoundError(
        f"Plugin directory not found at {PKG_DIR}. "
        "Please ensure the repository was cloned correctly."
    )

if shutil.which("git") is None and IN_COLAB:
    raise RuntimeError("git is required but unavailable in this runtime.")

# --------------------------------------------------------------------------
# 3. Dependency Management
# --------------------------------------------------------------------------

if REQ_FILE.exists():
    keep = []
    # These packages are managed by Colab's pre-installed environment.
    # Installing them again can cause conflicts.
    blocked = {"vllm", "torch", "torchvision", "torchaudio", "numpy", "scipy", "protobuf"}

    for line in REQ_FILE.read_text().splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith("#"):
            keep.append(line)
            continue

        package = package_from_req_line(stripped)
        if package in blocked:
            print(f"⏭ Skipping managed dependency: {line}")
            continue
        keep.append(line)

    FILTERED_REQ_FILE.write_text("\n".join(keep) + "\n")
    print(f"Installing filtered requirements from {REQ_FILE.name}...")
    run([sys.executable, "-m", "pip", "install", "-r", str(FILTERED_REQ_FILE)])
else:
    print("⚠ Warning: No requirement.txt found; skipping custom dependency installation.")

# Ensure protobuf is at the correct version
print("Ensuring protobuf version compatibility...")
run([sys.executable, "-m", "pip", "install", "--force-reinstall", "protobuf>=5.29.6,<6.30"])

# --------------------------------------------------------------------------
# 4. vLLM & PyTorch Installation
# --------------------------------------------------------------------------

if COLAB_INSTALL_VLLM:
    print(f"Installing user-specified vLLM: {COLAB_INSTALL_VLLM}")
    run([sys.executable, "-m", "pip", "install", COLAB_INSTALL_VLLM])
else:
    print(f"\nInstalling vLLM and PyTorch for {VLLM_TORCH_BACKEND}...")

    # 1. Uninstall existing conflicting versions
    run([sys.executable, "-m", "pip", "uninstall", "-y", "vllm", "torch", "torchvision", "torchaudio"])

    # 2. Manually remove leftover binary artifacts that pip might miss
    for site_dir in site.getsitepackages():
        site_path = Path(site_dir)
        leftovers = [
            site_path / "vllm", *site_path.glob("vllm-*.dist-info"),
            site_path / "torch", *site_path.glob("torch-*.dist-info"),
            site_path / "torchvision", *site_path.glob("torchvision-*.dist-info"),
            site_path / "torchaudio", *site_path.glob("torchaudio-*.dist-info"),
        ]
        for leftover in leftovers:
            if leftover.exists():
                print(f"  Cleaning up leftover: {leftover.name}")
                if leftover.is_dir():
                    shutil.rmtree(leftover)
                else:
                    leftover.unlink()

    # 3. Force clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
    except:
        pass

    # 4. Install using 'uv' (faster than pip) with specific CUDA backend
    print("  Downloading and installing packages with 'uv'...")
    run([sys.executable, "-m", "pip", "install", "-U", "uv"])
    run([
        "uv", "pip", "install",
        "--system",
        VLLM_SPEC,
        "torch", "torchvision", "torchaudio",
        f"--torch-backend={VLLM_TORCH_BACKEND}",
    ])

    # 5. Verify Scipy/Numpy
    scipy_check = run_capture([
        sys.executable, "-c", "import numpy, scipy; print('numpy', numpy.__version__); print('scipy', scipy.__version__)"
    ])
    if scipy_check.returncode:
        print("  Detected numpy/scipy issues; upgrading...")
        run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "numpy", "scipy"])

# Verify Installation
print("\nVerifying installation...")
run([
    sys.executable,
    "-c",
    "import torch, vllm; print('✓ torch:', torch.__version__); print('✓ vllm:', getattr(vllm, '__version__', 'unknown'))"
])

# --------------------------------------------------------------------------
# 5. Plugin Loading
# --------------------------------------------------------------------------

# Strategy: Add path to sys.path to allow immediate import,
# then ensure metadata is installed for subprocesses later.
plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)
importlib.invalidate_caches()

try:
    spec = importlib.util.spec_from_file_location("vllm_hook_plugins", PKG_DIR / "__init__.py")
    if spec:
        importlib.util.module_from_spec(spec)
        print("✓ Plugin module loaded successfully from source path.")
except Exception as e:
    print(f"⚠ Warning: Initial import check failed ({e}). This is expected if compilation is needed later.")

# Ensure the package is registered in the environment (metadata)
# This is necessary for tools that rely on `importlib.metadata`.
print("Registering plugin package in environment...")
run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PKG_DIR)])

print("Installing W&B tracking client...")
run([sys.executable, "-m", "pip", "install", "wandb"])

print(f"Plugin Source: {plugin_src_dir}")
print(f"Python Exec  : {sys.executable}")

# --------------------------------------------------------------------------
# 6. Runtime Restart (Critical for Colab)
# --------------------------------------------------------------------------
# We restart the runtime to ensure the newly installed binary libraries
# are loaded into a fresh Python interpreter. This prevents "dirty state" issues.
if IN_COLAB and not COLAB_RESTART_MARKER.exists():
    COLAB_RESTART_MARKER.write_text("1\n")
    print("\n" + "="*50)
    print("RESTARTING COLAB RUNTIME...")
    print("This ensures the new vLLM/Torch binaries are fully loaded.")
    print("Do not interrupt this process.")
    print("="*50)

    time.sleep(1) # Allow output buffer to flush
    sys.exit(0) # Terminate this process to trigger Colab restart


Cloning https://github.com/tburleyinfo/vLLM-Hook.git (branch: codex/MLR-45-cnl-spotlight-blocking-colab)...
> Running: git clone --branch codex/MLR-45-cnl-spotlight-blocking-colab https://github.com/tburleyinfo/vLLM-Hook.git /content/vLLM-Hook
Cloning into '/content/vLLM-Hook'...
fatal: Remote branch codex/MLR-45-cnl-spotlight-blocking-colab not found in upstream origin


RuntimeError: Command failed with exit code 128: git clone --branch codex/MLR-45-cnl-spotlight-blocking-colab https://github.com/tburleyinfo/vLLM-Hook.git /content/vLLM-Hook

Last output lines:


### Scoring Dependencies

The E-Prime checker uses spaCy for state-of-being verb detection. Run this cell before scoring.


In [ ]:
# Optional scoring dependency setup for Colab.
# Run after the main setup cell. If you already have spaCy and en_core_web_sm, this is a no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("spacy") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "spacy"])

try:
    import spacy
    spacy.load("en_core_web_sm", disable=["ner", "parser"])
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


### Imports & Environment


In [ ]:
import gc
import io
import json
import os
import multiprocessing as mp
import re
import subprocess
import sys
import time
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch
from vllm import SamplingParams

if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

from vllm_hook_plugins import HookLLM, generate_with_spotlight, register_plugins
from vllm_hook_plugins.logits_processors import default_eprime_bad_words
from research.experiment_tracking import (
    ExperimentCondition,
    ExperimentResult,
    RunMetrics,
    TurnResult,
    WandbTracker,
    evaluate_mlr20_condition,
    log_mlr20_results_sequentially,
    mlr20_alpha_sweep_conditions,
    run_mlr20_batch,
    collect_runtime_provenance,
    format_gib,
    has_required_vram,
    required_vram_bytes,
    turn_result_from_eprime_row,
)

IN_COLAB = "google.colab" in sys.modules
os.environ["VLLM_USE_V1"] = "1"

if IN_COLAB:
    mp.set_start_method("fork", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "fork"
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    os.environ.setdefault("HF_HOME", "/content/.cache/huggingface")
    os.environ.setdefault("HUGGINGFACE_HUB_CACHE", "/content/.cache/huggingface/hub")
    os.makedirs(os.environ["HUGGINGFACE_HUB_CACHE"], exist_ok=True)

    def _patch_fileno(stream, fallback_stream, fallback_fd):
        try:
            stream.fileno()
        except io.UnsupportedOperation:
            def _fileno():
                try:
                    return fallback_stream.fileno()
                except Exception:
                    return fallback_fd
            stream.fileno = _fileno

    _patch_fileno(sys.stdout, sys.__stdout__, 1)
    _patch_fileno(sys.stderr, sys.__stderr__, 2)
else:
    mp.set_start_method("spawn", force=True)
    os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

register_plugins()
print("Environment configured")


### GPU Memory Preflight


In [ ]:
def show_nvidia_smi():
    if not IN_COLAB:
        return
    try:
        completed = subprocess.run(
            ["nvidia-smi"],
            check=False,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        print("nvidia-smi unavailable in this runtime.")
        return
    if completed.stdout.strip():
        print(completed.stdout)
    if completed.stderr.strip():
        print(completed.stderr)


def cuda_memory_preflight(gpu_memory_utilization, *, show_smi=True):
    gc.collect()
    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable. In Colab, select a GPU runtime, restart the runtime, "
            "and run the notebook from the beginning."
        )

    torch.cuda.empty_cache()
    if hasattr(torch.cuda, "ipc_collect"):
        try:
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"torch.cuda.ipc_collect() skipped: {exc}")

    free_bytes, total_bytes = torch.cuda.mem_get_info()
    required_bytes = required_vram_bytes(total_bytes, gpu_memory_utilization)
    print(
        "CUDA memory after cleanup: "
        f"free={format_gib(free_bytes)} total={format_gib(total_bytes)} "
        f"required={format_gib(required_bytes)} "
        f"for gpu_memory_utilization={gpu_memory_utilization:.2f}"
    )

    if show_smi:
        show_nvidia_smi()

    if not has_required_vram(
        free_bytes=free_bytes,
        total_bytes=total_bytes,
        gpu_memory_utilization=gpu_memory_utilization,
    ):
        raise RuntimeError(
            "Not enough free CUDA memory for the configured vLLM reservation. "
            f"Free={format_gib(free_bytes)}, required={format_gib(required_bytes)}, "
            f"total={format_gib(total_bytes)}, "
            f"gpu_memory_utilization={gpu_memory_utilization:.2f}. "
            "Restart the Colab runtime and choose Runtime > Run all. "
            "Do not continue from a partially executed runtime."
        )

    return {
        "free_bytes": free_bytes,
        "total_bytes": total_bytes,
        "required_bytes": required_bytes,
    }


### Initialize `HookLLM`


In [ ]:
cache_dir = "/content/.cache/vllm-hook" if IN_COLAB else os.path.expanduser("~/.cache/vllm-hook")
model = "Qwen/Qwen2-1.5B-Instruct"
MAX_MODEL_LEN = 8192
GPU_MEMORY_UTILIZATION = float(os.environ.get("VLLM_HOOK_GPU_MEMORY_UTILIZATION", "0.30" if IN_COLAB else "0.65"))

gpu_memory_preflight = cuda_memory_preflight(GPU_MEMORY_UTILIZATION)

llm = HookLLM(
    model=model,
    worker_name="probe_spotlight",
    download_dir=cache_dir,
    trust_remote_code=True,
    dtype=torch.float16,
    enable_hook=True,
    gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
    max_model_len=MAX_MODEL_LEN,
    max_num_seqs=1,
    enforce_eager=True,
    enable_prefix_caching=False,
    enable_chunked_prefill=False,
    tensor_parallel_size=1,
)

print(f"Model loaded: {model}")
print("Spotlight worker enabled")
print(f"Max model length: {MAX_MODEL_LEN}")
print(f"GPU memory utilization: {GPU_MEMORY_UTILIZATION}")


### E-Prime And Possibility E-Prime Configuration

This notebook tests two simple CNLs:

- E-Prime: logit blocking only. It removes state-of-being verbs and related contractions.
- Possibility E-Prime: Spotlight plus logit blocking. It uses the same E-Prime blocking and promotes the rule that every action verb should follow `may`, `might`, `can`, or `could`.

For E-Prime adherence, this notebook deliberately blocks apostrophe-s forms. That can remove possessives too, but it keeps the surface rule strict.

In [ ]:
STATE_OF_BEING_FORMS = [
    "am",
    "is",
    "are",
    "was",
    "were",
    "be",
    "being",
    "been",
]

# Token-level bad-word coverage for common tokenizer segmentations.
# vLLM's native bad_words processor tokenizes these strings internally.
# Apostrophe-s is included for strict E-Prime adherence, even though it also
# catches possessives such as "John's book".
EPRIME_CONTRACTION_TOKEN_PATTERNS = [
    "'m",
    "’m",
    "'re",
    "’re",
    "'s",
    "’s",
]

# Optional broader contraction suffixes. These are not state-of-being forms,
# but they help if the CNL says "no contractions" rather than only E-Prime.
GENERAL_CONTRACTION_TOKEN_PATTERNS = [
    "n't",
    "n’t",
    "'ve",
    "’ve",
    "'ll",
    "’ll",
    "'d",
    "’d",
]

POSSIBILITY_AUXILIARIES = ["may", "might", "can", "could"]


def eprime_bad_words(*, include_general_contractions=False):
    bad_words = []
    bad_words.extend(STATE_OF_BEING_FORMS)
    bad_words.extend(EPRIME_CONTRACTION_TOKEN_PATTERNS)
    if include_general_contractions:
        bad_words.extend(GENERAL_CONTRACTION_TOKEN_PATTERNS)
    return sorted({word for word in bad_words if word})


CNL_RULESETS = {
    "e_prime": {
        "name": "E-Prime",
        "use_spotlight": False,
        "use_bad_words": True,
        "instruction": """Controlled Natural Language rules:
Use E-Prime.
Do not use state-of-being verbs: am, is, are, was, were, be, being, or been.
Do not use contractions that contain state-of-being verbs.
Do not use apostrophe-s forms.
""".strip(),
        "bad_words": eprime_bad_words(include_general_contractions=False),
        "possibility_auxiliaries": [],
    },
    "possibility_e_prime": {
        "name": "Possibility E-Prime",
        "use_spotlight": True,
        "use_bad_words": True,
        "instruction": """Controlled Natural Language rules:
Use E-Prime.
Do not use state-of-being verbs: am, is, are, was, were, be, being, or been.
Do not use contractions that contain state-of-being verbs.
Do not use apostrophe-s forms.
For every action verb, place one auxiliary directly before it: may, might, can, or could.
""".strip(),
        "bad_words": eprime_bad_words(include_general_contractions=False),
        "possibility_auxiliaries": POSSIBILITY_AUXILIARIES,
    },
}

SYSTEM_MESSAGE = (
    "You are a concise assistant. Follow the Controlled Natural Language rules "
    "while still answering the user helpfully."
)

PROMPTS = [
    "Explain why people should back up important files.",
    "Explain how a city can reduce traffic without making daily life harder.",
    "Explain why a model might follow instructions at first and then drift later.",
    "Explain how a student can check whether an online source deserves trust.",
]

ALPHA = 0.2
SAMPLING_PARAMS = SamplingParams(temperature=0.0, max_tokens=140)

for rule_id, ruleset in CNL_RULESETS.items():
    print(f"{rule_id}: {ruleset['name']}")
    print(f"  Spotlight: {ruleset['use_spotlight']}")
    print(f"  Bad words: {len(ruleset['bad_words'])}")
    print(f"  Auxiliaries: {ruleset['possibility_auxiliaries'] or 'N/A'}")

### Prompt, Sampling, And Generation Helpers

Each prompt runs once per CNL. E-Prime uses blocking only. Possibility E-Prime uses Spotlight plus the same blocking list.

In [ ]:
def render_cnl_prompt(user_prompt, ruleset):
    return f"""{SYSTEM_MESSAGE}

{ruleset['instruction']}

USER: {user_prompt}
ASSISTANT:""".strip()


def sampling_params_with_bad_words(base_params=SAMPLING_PARAMS, bad_words=None):
    bad_words = list(bad_words or [])
    return SamplingParams(
        temperature=base_params.temperature,
        max_tokens=base_params.max_tokens,
        bad_words=bad_words,
    )


def extract_text(outputs):
    return outputs[0].outputs[0].text.strip()


def generate_for_ruleset(prompt, ruleset):
    params = SAMPLING_PARAMS
    if ruleset.get("use_bad_words"):
        params = sampling_params_with_bad_words(bad_words=ruleset["bad_words"])

    if ruleset.get("use_spotlight"):
        outputs = generate_with_spotlight(
            llm,
            prompts=[prompt],
            emph_strings=[ruleset["instruction"]],
            alpha=ALPHA,
            sampling_params=params,
        )
    else:
        outputs = llm.generate([prompt], sampling_params=params, use_hook=False)
    return extract_text(outputs)


rendered_prompts = {
    rule_id: [render_cnl_prompt(prompt, ruleset) for prompt in PROMPTS]
    for rule_id, ruleset in CNL_RULESETS.items()
}
print(rendered_prompts["e_prime"][0])

### spaCy CNL Evaluator

The evaluator scores assistant replies only. It does not inspect the prompt text.

E-Prime checks for blocked surface forms and spaCy state-of-being verbs. Possibility E-Prime adds one extra rule: every action verb must have `may`, `might`, `can`, or `could` immediately before it.

In [ ]:
import spacy

CNL_NLP = spacy.load("en_core_web_sm", disable=["ner"])
EPRIME_CONTRACTION_RE = re.compile(
    r"\b\w+(?:['’]m|['’]re|['’]s)\b",
    re.IGNORECASE,
)
STATE_OF_BEING_SET = {word.lower() for word in STATE_OF_BEING_FORMS}


def _lower_set(items):
    return {item.lower() for item in items}


def is_state_of_being_token(token):
    text = token.text.lower()
    lemma = token.lemma_.lower()
    if text in STATE_OF_BEING_SET and token.pos_ in {"AUX", "VERB"}:
        return True
    return lemma == "be" and token.pos_ in {"AUX", "VERB"}


def _token_match_record(token):
    return {
        "text": token.text,
        "lemma": token.lemma_,
        "pos": token.pos_,
        "tag": token.tag_,
        "start": token.idx,
        "end": token.idx + len(token.text),
    }


def previous_word_token(doc, token_index):
    for j in range(token_index - 1, -1, -1):
        candidate = doc[j]
        if not candidate.is_space and not candidate.is_punct:
            return candidate
    return None


def action_verb_without_possibility_aux(token, doc, auxiliaries):
    if token.pos_ not in {"VERB", "AUX"}:
        return False
    if is_state_of_being_token(token):
        return False
    if token.lemma_.lower() in auxiliaries:
        return False
    previous = previous_word_token(doc, token.i)
    if previous is None:
        return True
    return previous.text.lower() not in auxiliaries


def evaluate_cnl_text(text, ruleset, nlp=CNL_NLP):
    # Pass only assistant reply text here. Prompt text should never be scored.
    text = text or ""
    doc = nlp(text)
    bad_words = _lower_set(ruleset["bad_words"])
    auxiliaries = _lower_set(ruleset.get("possibility_auxiliaries", []))

    blocked_matches = []
    state_of_being_matches = []
    for token in doc:
        token_text = token.text.lower()
        if token_text in bad_words:
            blocked_matches.append(_token_match_record(token))
        if is_state_of_being_token(token):
            state_of_being_matches.append(_token_match_record(token))

    contraction_matches = [
        {"text": match.group(0), "start": match.start(), "end": match.end()}
        for match in EPRIME_CONTRACTION_RE.finditer(text)
    ]

    auxiliary_verb_violations = []
    if auxiliaries:
        for token in doc:
            if action_verb_without_possibility_aux(token, doc, auxiliaries):
                record = _token_match_record(token)
                previous = previous_word_token(doc, token.i)
                record["previous_token"] = None if previous is None else previous.text
                auxiliary_verb_violations.append(record)

    violation_count = (
        len(blocked_matches)
        + len(state_of_being_matches)
        + len(contraction_matches)
        + len(auxiliary_verb_violations)
    )

    return {
        "blocked_word_count": len(blocked_matches),
        "state_of_being_count": len(state_of_being_matches),
        "contraction_count": len(contraction_matches),
        "auxiliary_verb_violation_count": len(auxiliary_verb_violations),
        "violation_count": violation_count,
        "cnl_retained": violation_count == 0,
        "blocked_matches": blocked_matches,
        "state_of_being_matches": state_of_being_matches,
        "contraction_matches": contraction_matches,
        "auxiliary_verb_violations": auxiliary_verb_violations,
    }


def score_reply(rule_id, prompt_index, user_prompt, rendered_prompt, reply):
    ruleset = CNL_RULESETS[rule_id]
    score = evaluate_cnl_text(reply, ruleset)
    return {
        "rule_id": rule_id,
        "condition": ruleset["name"],
        "spotlight_on": ruleset["use_spotlight"],
        "logit_blocking_on": ruleset["use_bad_words"],
        "prompt_index": prompt_index,
        "user_prompt": user_prompt,
        "prompt": rendered_prompt,
        "reply": reply,
        **score,
    }

print(evaluate_cnl_text("People may save files. The backup is useful.", CNL_RULESETS["possibility_e_prime"]))

### Run E-Prime And Possibility E-Prime

Set `RUN_FULL_PROMPT_SET = False` for a quick single-prompt smoke test.

In [ ]:
RUN_FULL_PROMPT_SET = True
active_prompts = PROMPTS if RUN_FULL_PROMPT_SET else PROMPTS[:1]
rows = []

for rule_id, ruleset in CNL_RULESETS.items():
    for prompt_index, user_prompt in enumerate(active_prompts):
        rendered_prompt = render_cnl_prompt(user_prompt, ruleset)
        print(f"Running {rule_id} prompt {prompt_index}...")
        reply = generate_for_ruleset(rendered_prompt, ruleset)
        rows.append(score_reply(rule_id, prompt_index, user_prompt, rendered_prompt, reply))

results = pd.DataFrame(rows)
results[[
    "rule_id",
    "condition",
    "spotlight_on",
    "logit_blocking_on",
    "prompt_index",
    "blocked_word_count",
    "state_of_being_count",
    "contraction_count",
    "auxiliary_verb_violation_count",
    "violation_count",
    "cnl_retained",
]]

### Compare Composition Adherence

E-Prime adherence comes from blocking state-of-being surface forms. Possibility E-Prime adherence adds the auxiliary-before-action-verb check.

In [ ]:
composition_summary = results.groupby("rule_id").agg(
    condition=("condition", "first"),
    prompts=("prompt_index", "count"),
    spotlight_on=("spotlight_on", "first"),
    logit_blocking_on=("logit_blocking_on", "first"),
    retained_rate=("cnl_retained", "mean"),
    violations=("violation_count", "sum"),
    blocked_words=("blocked_word_count", "sum"),
    state_of_being=("state_of_being_count", "sum"),
    contractions=("contraction_count", "sum"),
    auxiliary_verb_violations=("auxiliary_verb_violation_count", "sum"),
).sort_values(["retained_rate", "violations"], ascending=[False, True])
composition_summary

### Inspect Assistant Replies And Failures

This cell prints only generated assistant replies and their adherence counts.

In [ ]:
for _, row in results.iterrows():
    print("=" * 88)
    print(f"Rule: {row['rule_id']} | Prompt {row['prompt_index']}")
    print(f"Spotlight: {row['spotlight_on']} | Logit blocking: {row['logit_blocking_on']}")
    print(f"User prompt: {row['user_prompt']}")
    print("-" * 88)
    print(row["reply"])
    print("-" * 88)
    print(
        "violations=", row["violation_count"],
        "blocked=", row["blocked_word_count"],
        "state_of_being=", row["state_of_being_count"],
        "contractions=", row["contraction_count"],
        "auxiliary_verb=", row["auxiliary_verb_violation_count"],
    )

failures = results[results["violation_count"] > 0][[
    "rule_id",
    "prompt_index",
    "violation_count",
    "blocked_matches",
    "state_of_being_matches",
    "contraction_matches",
    "auxiliary_verb_violations",
    "reply",
]]
failures

### Save Results

In [ ]:
out_dir = Path("/content/cnl_spotlight_blocking_results") if IN_COLAB else Path("notebooks/results")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")
results_path = out_dir / f"mlr45_cnl_spotlight_blocking_{stamp}.jsonl"
summary_path = out_dir / f"mlr45_cnl_spotlight_blocking_summary_{stamp}.csv"
results.to_json(results_path, orient="records", lines=True)
composition_summary.to_csv(summary_path)
print(f"Saved rows to {results_path}")
print(f"Saved summary to {summary_path}")
print("Validation note: run this notebook in a Colab GPU runtime for execution results.")